# Qwen2.5-VL-3B first-token LM-head baseline

This notebook classifies the MELD **test split** without training an MLP and without decoding videos again. It applies Qwen's frozen language-model head to the saved final shared video+utterance hidden states, then selects the highest-logit first token among `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, and `surprise`.

This is deliberately a **first-token verbalizer** experiment. If a label has multiple tokenizer pieces, only its first contextual token is scored. The notebook audits and saves those tokenizations before evaluation.

## 1. Paths and fixed experiment settings

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
EMBEDDINGS_PT = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/embeddings/meld_test_clean.pt"
RUN_ROOT = Path("/kaggle/working/experiments/baselines/qwen2_5_vl_3b_lm_head_first_token/test")
BATCH_SIZE = 64
LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

RUN_ROOT.mkdir(parents=True, exist_ok=True)
assert EMBEDDINGS_PT.is_file(), f"Missing saved test embeddings: {EMBEDDINGS_PT}"
print("Embeddings:", EMBEDDINGS_PT)
print("Outputs:", RUN_ROOT)

## 2. Install the tested runtime

Keep Kaggle's CUDA-enabled PyTorch. Restart the session if a different Transformers version was already imported.

In [ ]:
!pip install -q "transformers==4.57.1" accelerate pandas scikit-learn tqdm

In [ ]:
import gc
import json
import math
import os
import shutil

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import FileLink, display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

assert transformers.__version__ == "4.57.1", "Restart the Kaggle session after installing Transformers 4.57.1."
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Transformers:", transformers.__version__)
print("Scoring device:", DEVICE)
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

## 3. Load and validate the saved test embeddings

The LM-head projection is valid only for final-layer, last-token embeddings produced with Qwen's assistant generation marker. These checks prevent accidentally scoring mean-pooled, earlier-layer, ablated, or differently prompted embeddings.

In [ ]:
payload = torch.load(EMBEDDINGS_PT, map_location="cpu", weights_only=False)
embeddings = payload["embeddings"].float()
true_ids = payload["labels"].long()
sample_ids = [str(value) for value in payload["sample_ids"]]
metadata = payload.get("metadata", [])
label_names = list(payload.get("label_names") or LABELS)

assert label_names == LABELS, (label_names, LABELS)
assert embeddings.ndim == 2 and embeddings.shape[1] == 2048, embeddings.shape
assert embeddings.shape[0] == true_ids.shape[0] == len(sample_ids)
assert not metadata or len(metadata) == len(sample_ids)
assert len(set(sample_ids)) == len(sample_ids), "Duplicate sample IDs found"

source_sample_count = len(sample_ids)
finite_mask = torch.isfinite(embeddings).all(dim=1)
dropped_indices = (~finite_mask).nonzero(as_tuple=True)[0].tolist()
dropped_non_finite = pd.DataFrame({
    "row_index": dropped_indices,
    "sample_id": [sample_ids[index] for index in dropped_indices],
})
dropped_non_finite.to_csv(RUN_ROOT / "dropped_non_finite_embeddings.csv", index=False)
if dropped_indices:
    keep = finite_mask.tolist()
    embeddings = embeddings[finite_mask]
    true_ids = true_ids[finite_mask]
    sample_ids = [value for value, retain in zip(sample_ids, keep) if retain]
    if metadata:
        metadata = [value for value, retain in zip(metadata, keep) if retain]

assert torch.isfinite(embeddings).all(), "Non-finite rows remain after filtering"
assert embeddings.shape[0] == true_ids.shape[0] == len(sample_ids)
assert not metadata or len(metadata) == len(sample_ids)
assert true_ids.min().item() >= 0 and true_ids.max().item() < len(LABELS)

stored_config = payload.get("config", {})
if "merged_from" in stored_config:
    chunk_configs = [entry.get("config", {}) for entry in stored_config["merged_from"]]
else:
    chunk_configs = [stored_config]
assert chunk_configs, "No extraction configuration found"

def recorded_values(key):
    return [config[key] for config in chunk_configs if key in config]

def require_recorded_value(key, expected_value):
    values = recorded_values(key)
    assert values, f"Legacy payload has no recorded evidence for required field {key!r}"
    conflicts = [value for value in values if value != expected_value]
    assert not conflicts, f"Incompatible extraction config: {key} contains {conflicts!r}, expected {expected_value!r}"

def reject_recorded_conflicts(key, expected_value):
    conflicts = [value for value in recorded_values(key) if value != expected_value]
    assert not conflicts, f"Incompatible extraction config: {key} contains {conflicts!r}, expected {expected_value!r}"

# Seeded legacy chunks omit most fields; later chunks preserve positive provenance.
require_recorded_value("model_id", MODEL_ID)
require_recorded_value("layer", -1)
require_recorded_value("pooling", "last")
require_recorded_value("prompt_style", "emotion_task")
require_recorded_value("no_generation_prompt", False)

# These fields were introduced later. Missing means the historical baseline default;
# any explicitly recorded conflicting value still stops the experiment.
reject_recorded_conflicts("modality_mode", "video_text")
reject_recorded_conflicts("visual_token_ablation", "none")
reject_recorded_conflicts("utterance_token_ablation", "none")
reject_recorded_conflicts("visual_token_pruning", "none")
legacy_config_count = sum(not config.get("model_id") for config in chunk_configs)

print("Source samples:", source_sample_count)
print("Dropped non-finite rows:", len(dropped_indices))
print("Scored samples:", len(sample_ids))
print("Chunk configs:", len(chunk_configs), f"({legacy_config_count} legacy seed records)")
print("Embedding tensor:", tuple(embeddings.shape), embeddings.dtype)
print("Stored extraction errors:", len(payload.get("errors", [])))

## 4. Audit contextual emotion-token verbalizers

The candidate is appended directly after Qwen's assistant-generation prefix, matching the location represented by each saved last-token embedding. Classification can proceed with multi-token labels only when their first contextual token IDs are distinct.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
probe_messages = [{"role": "user", "content": [{"type": "text", "text": "Classify the emotion."}]}]
assistant_prefix = processor.apply_chat_template(
    probe_messages, tokenize=False, add_generation_prompt=True
)
prefix_ids = tokenizer(assistant_prefix, add_special_tokens=False)["input_ids"]

token_rows = []
for emotion in LABELS:
    full_ids = tokenizer(assistant_prefix + emotion, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prefix_ids)] == prefix_ids, f"Tokenizer boundary changed for {emotion}"
    candidate_ids = full_ids[len(prefix_ids):]
    assert candidate_ids, f"No candidate tokens for {emotion}"
    token_rows.append({
        "emotion": emotion,
        "token_count": len(candidate_ids),
        "token_ids": json.dumps(candidate_ids),
        "tokens": json.dumps(tokenizer.convert_ids_to_tokens(candidate_ids), ensure_ascii=False),
        "first_token_id": int(candidate_ids[0]),
        "first_token": tokenizer.convert_ids_to_tokens(int(candidate_ids[0])),
    })

token_audit = pd.DataFrame(token_rows)
first_token_ids = token_audit["first_token_id"].astype(int).tolist()
assert len(set(first_token_ids)) == len(LABELS), "Two emotion labels share a first token and cannot be distinguished"
token_audit.to_csv(RUN_ROOT / "emotion_tokenization.csv", index=False)
display(token_audit)
if (token_audit.token_count > 1).any():
    print("NOTE: Multi-token labels are intentionally scored by their first token only.")
else:
    print("All seven labels are single contextual tokens.")

## 5. Load Qwen and retain only its frozen LM head

The full checkpoint is loaded on CPU only long enough to obtain the original LM head. The multimodal transformer and vision encoder are then deleted. No model forward pass over videos is performed.

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True,
)
assert model.lm_head.in_features == embeddings.shape[1]
assert max(first_token_ids) < model.lm_head.out_features
model_revision = getattr(model.config, "_commit_hash", None)
lm_head = model.lm_head
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
lm_head = lm_head.to(DEVICE).eval()
print("LM head:", lm_head.in_features, "->", lm_head.out_features)
print("Checkpoint revision:", model_revision)

## 6. Score the complete test split

`raw_vocab_prob_*` is the token probability under Qwen's full vocabulary. `restricted_prob_*` renormalizes only the seven candidate-token logits and is used for forced-choice prediction. `candidate_probability_mass` shows how much raw probability Qwen assigned to all seven emotion first tokens together.

In [ ]:
token_id_tensor = torch.tensor(first_token_ids, dtype=torch.long, device=DEVICE)
all_selected_logits = []
all_raw_probs = []
all_restricted_probs = []
all_vocab_top_ids = []
all_vocab_top_probs = []

with torch.inference_mode():
    for start in tqdm(range(0, len(embeddings), BATCH_SIZE), desc="LM-head scoring"):
        batch = embeddings[start:start + BATCH_SIZE].to(DEVICE, dtype=lm_head.weight.dtype)
        vocabulary_logits = lm_head(batch).float()
        selected_logits = vocabulary_logits.index_select(1, token_id_tensor)
        vocabulary_log_probs = torch.log_softmax(vocabulary_logits, dim=-1)
        raw_probs = vocabulary_log_probs.index_select(1, token_id_tensor).exp()
        restricted_probs = torch.softmax(selected_logits, dim=-1)
        top_logits, top_ids = vocabulary_logits.max(dim=-1)
        top_probs = (top_logits - torch.logsumexp(vocabulary_logits, dim=-1)).exp()

        all_selected_logits.append(selected_logits.cpu())
        all_raw_probs.append(raw_probs.cpu())
        all_restricted_probs.append(restricted_probs.cpu())
        all_vocab_top_ids.append(top_ids.cpu())
        all_vocab_top_probs.append(top_probs.cpu())
        del batch, vocabulary_logits, vocabulary_log_probs

selected_logits = torch.cat(all_selected_logits).numpy()
raw_probs = torch.cat(all_raw_probs).numpy()
restricted_probs = torch.cat(all_restricted_probs).numpy()
vocab_top_ids = torch.cat(all_vocab_top_ids).numpy()
vocab_top_probs = torch.cat(all_vocab_top_probs).numpy()
pred_ids = restricted_probs.argmax(axis=1)
sorted_probs = np.sort(restricted_probs, axis=1)
confidence = sorted_probs[:, -1]
margin = sorted_probs[:, -1] - sorted_probs[:, -2]
entropy = -(restricted_probs * np.log(np.clip(restricted_probs, 1e-12, 1.0))).sum(axis=1)
candidate_mass = raw_probs.sum(axis=1)
print("Scored samples:", len(pred_ids))

## 7. Save predictions and metrics

In [ ]:
true_np = true_ids.numpy()
predictions = pd.DataFrame({
    "sample_id": sample_ids,
    "true_id": true_np,
    "true_emotion": [LABELS[index] for index in true_np],
    "predicted_id": pred_ids,
    "predicted_emotion": [LABELS[index] for index in pred_ids],
    "correct": pred_ids == true_np,
    "restricted_confidence": confidence,
    "restricted_margin": margin,
    "restricted_entropy": entropy,
    "candidate_probability_mass": candidate_mass,
    "vocab_top_token_id": vocab_top_ids,
    "vocab_top_token": tokenizer.convert_ids_to_tokens(vocab_top_ids.tolist()),
    "vocab_top_probability": vocab_top_probs,
})
if metadata:
    for column in ["dialogue_id", "utterance_id", "utterance", "video_file"]:
        predictions[column] = [row.get(column) for row in metadata]
for index, emotion in enumerate(LABELS):
    predictions[f"logit_{emotion}"] = selected_logits[:, index]
    predictions[f"raw_vocab_prob_{emotion}"] = raw_probs[:, index]
    predictions[f"restricted_prob_{emotion}"] = restricted_probs[:, index]

cm = confusion_matrix(true_np, pred_ids, labels=list(range(len(LABELS))))
report = classification_report(
    true_np, pred_ids, labels=list(range(len(LABELS))), target_names=LABELS,
    output_dict=True, zero_division=0,
)
metrics = {
    "experiment": "qwen2_5_vl_3b_lm_head_first_token",
    "split": "test",
    "model_id": MODEL_ID,
    "model_revision": model_revision,
    "source_sample_count": source_sample_count,
    "dropped_non_finite_count": len(dropped_indices),
    "sample_count": len(predictions),
    "scoring_rule": "argmax over seven contextual first-token logits",
    "accuracy": float(accuracy_score(true_np, pred_ids)),
    "macro_f1": float(f1_score(true_np, pred_ids, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(true_np, pred_ids, average="weighted", zero_division=0)),
    "mean_restricted_confidence": float(confidence.mean()),
    "mean_restricted_margin": float(margin.mean()),
    "mean_restricted_entropy": float(entropy.mean()),
    "mean_candidate_probability_mass": float(candidate_mass.mean()),
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "label_names": LABELS,
    "first_token_ids": dict(zip(LABELS, first_token_ids)),
}

predictions.to_csv(RUN_ROOT / "test_predictions.csv", index=False)
pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]).to_csv(RUN_ROOT / "test_confusion_matrix.csv")
pd.DataFrame(report).T.to_csv(RUN_ROOT / "test_classification_report.csv")
(RUN_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
manifest = {
    "embeddings_path": str(EMBEDDINGS_PT),
    "source_sample_count": source_sample_count,
    "dropped_non_finite_count": len(dropped_indices),
    "scored_embedding_shape": list(embeddings.shape),
    "embedding_dtype_after_load": str(embeddings.dtype),
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "candidate_construction": "label appended directly after assistant generation prefix",
    "multi_token_policy": "first contextual token only",
}
(RUN_ROOT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({key: metrics[key] for key in ["accuracy", "macro_f1", "weighted_f1", "mean_candidate_probability_mass"]}, indent=2))

## 8. Inspect the result

In [ ]:
cm_display = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in LABELS],
    columns=[f"pred_{label}" for label in LABELS],
)
display(cm_display)
display(pd.DataFrame(report).T)
display(predictions.head(10))
print("Unrestricted vocabulary top token is one of the seven candidates:", predictions.vocab_top_token_id.isin(first_token_ids).mean())

## 9. Create a download bundle

The bundle contains tokenization, predictions, metrics, reports, and the run manifest. It excludes the source embeddings and model weights.

In [ ]:
archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print("Download bundle:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))